# TimeR4 — Tái hiện TRUNG THỰC + QLoRA (Wikidata-VI)

Bản này bám **mã nguồn released** của tác giả (`qianxinying/TimeR4`), với **thay đổi duy nhất**:
**full fine-tune LLaMA2-7B → QLoRA 4-bit**. Mọi thành phần khác giữ nguyên như tác giả.

## Bảng đối chiếu trung thực (faithfulness ledger)

| Thành phần | Tác giả (mã released) | Notebook này |
|---|---|---|
| LLM nền | LLaMA2-7B-chat | LLaMA2-7B-chat (mirror NousResearch) ✅ |
| **Fine-tune LLM** | **full fine-tune** | **QLoRA 4-bit** ← thay đổi DUY NHẤT |
| Retriever nền | all-mpnet-base-v2 (tiếng Anh) | all-mpnet-base-v2 ✅ |
| Fine-tune retriever | full-FT, ContrastiveTension, 2 epoch, lr 5e-5 | **full-FT** (không LoRA) ✅ |
| Special token retriever | `[B_SEP][A_SEP][F_SEP][L_SEP]` + 4017 `[Time_i]` | y hệt ✅ (4017 token inert với VI) |
| Negative | time / content(rel+entity) / both | bám `construct_negatives.py` ✅ |
| Retrieve | n=15, `re_rank=False` | n=15, `re_rank=False` ✅ |
| Rerank thời gian | KHÔNG chạy (main.py) | KHÔNG chạy ✅ (bỏ TEN + bỏ rerank qtype) |
| Rewrite | GPT-3.5-turbo-0125, temp=1.0, đúng prompt | y hệt ✅ (rewrite cả train như main.py) |
| Prompt reasoning / Alpaca | đúng main.py / train.py | y hệt ✅ |
| LLM: epoch / max_len | 3 / 1024 | 3 / 1024 ✅ |
| Metric Hit@1 | normalize + gold ⊆ toàn bộ output | y hệt ✅ |

## Hai điểm PHẢI khai báo trong khóa luận (không thể tránh)

1. **Retriever nền là mô hình chỉ-tiếng-Anh** (`all-mpnet-base-v2`). Giữ đúng tác giả nên
   nhúng tiếng Việt kém → retrieval yếu. Đây chính là bằng chứng phương pháp gốc không
   chuyển tốt sang tiếng Việt (biện minh cho việc v12 đổi sang retriever đa ngôn ngữ + Qwen).
2. **4017 token `[Time_i]`** của ICEWS được thêm y hệt tác giả nhưng **không xuất hiện**
   trong text fact/câu hỏi của Wikidata-VI → chúng là embedding "chết" (inert), không ảnh hưởng.

## Hai chỗ "vá" nhẹ so với mã released (để đúng mô tả bài báo, không đổi phương pháp)

- `retrival.py` vá 4 lỗi kỹ thuật để chạy được với ngày YYYY / YYYY-MM (mã gốc chỉ nhận YYYY-MM-DD).
- Lần retrieve thứ 2 dùng câu **đã rewrite** (đúng khung Retrieve-Rewrite-Retrieve của bài báo);
  main.py released có lỗi wiring khiến lần 2 dùng lại câu gốc.

> ⚠️ **BẢO MẬT:** không dán OpenAI API key vào code. Dùng **Colab Secrets** (ô cấu hình đã hỗ trợ).
> Nếu key cũ từng nằm trong notebook đẩy lên GitHub, hãy **thu hồi key đó ngay**.

> ⏱️ Thứ tự chạy: Cell 0 → 11. Retriever full-FT (~30–60′) + rewrite train (GPT-3.5, có cache)
> + QLoRA (~2–3h) + đánh giá toàn test. GPU L4/A100 (bf16) khuyến nghị.

In [ ]:
# ═══════════════════════════════════════════════════════════
# ⚙️ CẤU HÌNH — CHỈ SỬA Ô NÀY
# ═══════════════════════════════════════════════════════════
# Bản này TÁI HIỆN TRUNG THỰC TimeR4 (mã released của tác giả qianxinying/TimeR4).
# THAY ĐỔI DUY NHẤT: full fine-tune LLaMA2  →  QLoRA 4-bit.
# Retriever VẪN full fine-tune như tác giả (all-mpnet-base-v2, KHÔNG LoRA).
import os

# ⚠️ TUYỆT ĐỐI KHÔNG dán API key thẳng vào code (repo public sẽ lộ key).
# Cách khuyến nghị — Colab Secrets: thêm secret tên OPENAI_API_KEY, bật quyền cho notebook, rồi:
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('✅ Đã nạp OPENAI_API_KEY từ Colab Secrets')
except Exception as _e:
    if not os.environ.get('OPENAI_API_KEY'):
        print('⚠️ Chưa có OPENAI_API_KEY. Dùng Colab Secrets, hoặc đặt os.environ trước khi chạy,')
        print('   hoặc set USE_OPENAI_REWRITE=False (khi đó rewrite KHÔNG còn giống tác giả).')

# True  = rewrite bằng GPT-3.5-turbo-0125 (ĐÚNG tác giả) — cần key
# False = rewrite bằng LLaMA2 nội bộ (KHÔNG giống tác giả → phải khai báo trong khóa luận)
USE_OPENAI_REWRITE = True

# Rewrite luôn tập TRAIN? Tác giả CÓ (main.py rewrite cả train trước khi build prompt fine-tune).
#   True  = trung thực (tốn thêm API cho ~N_TRAIN câu; có cache vào Drive nên chỉ tốn 1 lần)
#   False = tiết kiệm nhưng train/test lệch phân phối (kém trung thực)
REWRITE_TRAIN = True

_key = os.environ.get('OPENAI_API_KEY', '')
if USE_OPENAI_REWRITE:
    assert _key.startswith('sk-'), '❌ Cần OPENAI_API_KEY thật (hoặc đặt USE_OPENAI_REWRITE=False).'
    print('✅ Rewrite = GPT-3.5-turbo-0125 (giống tác giả)')
else:
    print('⚠️ Rewrite = LLaMA2 nội bộ (KHÔNG giống tác giả)')

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai'], capture_output=True)
print('✅ openai sẵn sàng')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — DRIVE + LIB + LLaMA2-7B (4-bit) + QLoRA
#   ★ THAY ĐỔI DUY NHẤT so với tác giả: full-FT → QLoRA (4-bit NF4).
# ═══════════════════════════════════════════════════════════
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['WANDB_DISABLED'] = 'true'; os.environ['WANDB_MODE'] = 'disabled'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

try:
    from google.colab import drive
    drive.mount('/content/drive'); print('✅ Đã mount Drive')
except Exception as _e:
    print('ℹ️ Không phải Colab:', str(_e)[:80])

import subprocess, sys
subprocess.run([sys.executable,'-m','pip','uninstall','-y',
    'peft','sentence-transformers','transformers','huggingface-hub',
    'accelerate','wandb'], capture_output=True)
for pkg in ['huggingface_hub==0.23.4','peft==0.11.1','transformers==4.44.0',
            'sentence-transformers==3.3.1','accelerate==0.34.0','faiss-cpu','tqdm']:
    r = subprocess.run([sys.executable,'-m','pip','install','-q',pkg], capture_output=True, text=True)
    print(f'  {"✅" if r.returncode==0 else "❌"} {pkg}')
rb = subprocess.run([sys.executable,'-m','pip','install','-q','-U','bitsandbytes>=0.44'],
                    capture_output=True, text=True)
print(f'  {"✅" if rb.returncode==0 else "❌"} bitsandbytes')

import torch, json, glob, re, random, copy
from contextlib import nullcontext
from tqdm import tqdm
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, Trainer)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
print('✅ Import OK')

assert torch.cuda.is_available(), '⚠️ Cần GPU (L4/A100/T4).'
print('✅ GPU:', torch.cuda.get_device_name(0))
USE_BF16 = torch.cuda.is_bf16_supported()
_CDT = torch.bfloat16 if USE_BF16 else torch.float16
print(f'✅ compute dtype: {"bf16" if USE_BF16 else "fp16"}')
random.seed(42); torch.manual_seed(42)

# Tác giả: LLaMA2-7B-chat. Dùng mirror công khai (trọng số giống hệt, không cần token gated).
MODEL_NAME = 'NousResearch/Llama-2-7b-chat-hf'

print(f'\nLoading {MODEL_NAME} (4-bit NF4)...')
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=_CDT, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'   # như train.py tác giả

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map={'': 0},
    torch_dtype=_CDT, attn_implementation='eager')
llm.config.use_cache = False
for _k in ('temperature','top_p','top_k'):
    if hasattr(llm.generation_config, _k): setattr(llm.generation_config, _k, None)
import warnings as _w; _w.filterwarnings('ignore', message='.*do_sample.*')
print(f'✅ LLaMA2 (4-bit) loaded. VRAM {torch.cuda.memory_allocated()/1e9:.1f}GB')

# QLoRA adapter — ĐÂY là điểm thay full-FT. Target = mọi linear như QLoRA chuẩn.
llm = prepare_model_for_kbit_training(llm)
lora_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
llm = get_peft_model(llm, lora_cfg)
llm.print_trainable_parameters()
DEVICE = 'cuda'


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — CLONE REPO (lấy retrival.py) + TKG (kg/full.txt) + 15 CSV CÂU HỎI
# ═══════════════════════════════════════════════════════════
import pandas as pd, ast

REPO_DIR = '/content/TimeR4'
if not os.path.exists(REPO_DIR):
    os.system(f'git clone --depth 1 https://github.com/qianxinying/TimeR4.git {REPO_DIR}')
os.chdir(REPO_DIR)

DATA_ROOTS = ['/content/drive/MyDrive', '/content']

# ── TKG từ kg/full.txt (Wikidata VI) ──
kg_cands = []
for _root in DATA_ROOTS:
    kg_cands += glob.glob(f'{_root}/**/kg/full.txt', recursive=True)
    kg_cands += glob.glob(f'{_root}/**/full.txt', recursive=True)
assert kg_cands, '❌ Không thấy full.txt — upload folder kg/ lên Drive (MyDrive/tkgqa_data/kg/)'
KG_PATH = kg_cands[0]
triple_list = []
with open(KG_PATH, encoding='utf-8') as f:
    for line in f:
        line = line.rstrip('\n')
        if not line.strip(): continue
        parts = line.split('\t')
        if len(parts) >= 4:
            triple_list.append([p.strip() for p in parts[:4]])
print(f'✅ TKG: {KG_PATH}\n   {len(triple_list):,} bộ tứ | ví dụ: {triple_list[0]}')

# ── 15 file câu hỏi CSV ──
q_files = []
for _root in DATA_ROOTS:
    q_files += glob.glob(f'{_root}/**/*_merged.csv', recursive=True)
q_files = sorted(set(q_files))
assert q_files, '❌ Không thấy *_merged.csv — upload folder question_generated/ lên Drive'
print(f'Tìm thấy {len(q_files)} file câu hỏi')

def parse_answers(x):
    try:
        v = ast.literal_eval(x) if isinstance(x, str) else x
        return v if isinstance(v, list) else [str(v)]
    except Exception:
        return [str(x)]

rows = []
for qf in q_files:
    try: df = pd.read_csv(qf)
    except Exception as e: print('  ⚠️ lỗi đọc', qf, e); continue
    for _, r in df.iterrows():
        q = str(r.get('question', '')).strip()
        if not q or q.lower() == 'nan': continue
        rows.append({'question': q, 'answers': parse_answers(r.get('answers', '[]')),
                     'qtype': str(r.get('qtype','')), 'time_level': str(r.get('time_level','')),
                     'answer_type': str(r.get('answer_type',''))})
print(f'Tổng câu (thô): {len(rows):,}')

seen, uniq = set(), []
for r in rows:
    if r['question'] in seen: continue
    seen.add(r['question']); uniq.append(r)
print(f'Sau khử trùng lặp: {len(uniq):,}')

random.seed(42); random.shuffle(uniq)
split = int(len(uniq) * 0.8)
train_questions = uniq[:split]; test_questions = uniq[split:]
_ov = sum(1 for r in test_questions if r['question'] in {x['question'] for x in train_questions})
print(f'✅ Train: {len(train_questions):,} | Test: {len(test_questions):,} | Chồng lấn: {_ov}')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — PATCH retrival.py (chỉ vá lỗi kỹ thuật để CHẠY) + METRIC (khớp predict_answer.py)
#   ⛔ KHÔNG có TEN, KHÔNG có rerank qtype.
#      Tác giả main.py: re_rank=False và KHÔNG chuẩn hoá ngày trong câu hỏi
#      → grounding thời gian hoàn toàn nhờ bước rewrite GPT-3.5 (đúng như thiết kế gốc).
# ═══════════════════════════════════════════════════════════
with open('retrival.py') as f: code = f.read()
patches = [
    ('self.triplet_id_list = [[triple[0], triple[1], triple[2], triple[3]] for triple in id_list]',
     'self.triplet_id_list = [[triple[0], triple[1], triple[2], triple[3]] for triple in id_list] if id_list is not None else []'),
    ('self.model = SentenceTransformer(model_name, device="cuda")\n        self.embedding_size = embedding_size',
     'self.model = SentenceTransformer(model_name, device="cuda")\n        self.embedding_size = self.model.get_sentence_embedding_dimension()'),
    ('self.full_time = [datetime.strptime(triple[3], "%Y-%m-%d").date() for triple in triple_list]',
     'self.full_time = []\n        for triple in triple_list:\n            t = triple[3] if len(triple) > 3 else "2000-01-01"\n            if len(t) == 4: t += "-01-01"\n            elif len(t) == 7: t += "-01"\n            try: self.full_time.append(datetime.strptime(t[:10], "%Y-%m-%d").date())\n            except: self.full_time.append(datetime.strptime("2000-01-01", "%Y-%m-%d").date())'),
    ('corpus_embeddings = corpus_embeddings / np.linalg.norm(corpus_embeddings, axis=1)[:, None]',
     'if corpus_embeddings.ndim == 1:\n            corpus_embeddings = corpus_embeddings[np.newaxis, :]\n        corpus_embeddings = corpus_embeddings / np.linalg.norm(corpus_embeddings, axis=1)[:, None]'),
]
for old, new in patches:
    if old in code: code = code.replace(old, new)
with open('retrival.py', 'w') as f: f.write(code)
if 'retrival' in sys.modules: del sys.modules['retrival']
from retrival import Retrieval
print('✅ retrival.py patched (chỉ 4 lỗi kỹ thuật, KHÔNG đổi logic xếp hạng)')

def get_q(x): return x.get('question', x.get('Question', ''))
def get_qtype(x): return x.get('qtype', '')
def get_gold(x): return x.get('answer', x.get('answers', x.get('Answer', '?')))

import string as _string
def normalize(s):
    # ĐÚNG predict_answer.py: lower + bỏ dấu câu + bỏ a/an/the + bỏ <pad> + gộp khoảng trắng
    s = str(s).lower()
    s = ''.join(c for c in s if c not in set(_string.punctuation))
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = re.sub(r'\b(pad)\b', ' ', s)
    return ' '.join(s.split())

def evaluate_hit(pred_raw, gold):
    # Tác giả: ghép TOÀN BỘ output rồi kiểm tra gold (chuẩn hoá) ⊆ prediction (chuẩn hoá).
    golds = gold if isinstance(gold, list) else [gold]
    p = normalize(pred_raw)
    for g in golds:
        ng = normalize(g)
        if ng and ng in p: return True
    return False

def parse_list_answer(raw):
    m = re.findall(r"['\"]([^'\"]+)['\"]", raw)
    return m if m else [raw.strip('[]\'\" .')]

questions_1000 = test_questions
print('Tập test:', len(questions_1000))


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — DỰNG NEGATIVE (bám construct_negatives.py của tác giả)
#   query đào negative = "question answers" (như tác giả, để chắc fact gold lọt top-15)
#   positive        = triple retrieve top-1 (first/last: sắp theo ngày)
#   time-neg        : before/first → time+δ ; after/last → time−δ (δ=1..10 ngày)
#   content-neg     : giữ head+time, đổi relation VÀ entity-đáp-án (object)
#   both-neg        : đổi relation + entity + time
# ═══════════════════════════════════════════════════════════
from datetime import datetime, timedelta

all_entities  = list({t[0] for t in triple_list} | {t[2] for t in triple_list})
all_relations = list({t[1] for t in triple_list})
all_times     = list({t[3] for t in triple_list})

def triple_to_text(tr):
    # KHỚP định dạng fact trong retrival.py: '{s} {r} {o} in {t}.'
    return f'{tr[0]} {tr[1]} {tr[2]} in {tr[3]}.'

def _pdate(s):
    for fmt in ('%Y-%m-%d', '%Y-%m', '%Y'):
        try: return datetime.strptime(str(s)[:10], fmt).date()
        except Exception: pass
    return None

def _sort_key(tr):
    d = _pdate(tr[3]); return d if d else datetime.min.date()

def gold_answers(item):
    g = get_gold(item)
    if isinstance(g, str):
        inner = re.findall(r"['\"]([^'\"]+)['\"]", g); return inner if inner else [g]
    return g if isinstance(g, list) else [g]

def gen_negatives_author(pos, question_text, answers):
    s, r, o, t = pos[0], pos[1], pos[2], pos[3]
    base = _pdate(t); negs = []; ql = question_text.lower(); new_time_str = None
    if base is not None:
        if 'before' in ql or 'first' in ql: nt = base + timedelta(days=random.randint(1, 10))
        elif 'after' in ql or 'last' in ql: nt = base - timedelta(days=random.randint(1, 10))
        else: nt = None
        if nt is not None and nt.strftime('%Y-%m-%d') not in all_times:
            cand = [x for x in all_times if x != t]
            if cand:
                new_time_str = random.choice(cand)
                negs.append([s, r, o, new_time_str])                 # 1. time incorrect
    ans_low = set(str(a).lower() for a in answers)
    new_rel = random.choice([x for x in all_relations if x != r]) if len(all_relations) > 1 else r
    ent_pool = [x for x in all_entities if str(x).lower() not in ans_low]
    new_ent = random.choice(ent_pool) if ent_pool else o
    negs.append([s, new_rel, new_ent, t])                            # 2. content incorrect
    if new_time_str:
        negs.append([s, new_rel, new_ent, new_time_str])             # 3. both incorrect
    return negs

RETRIEVER_BASE = 'sentence-transformers/all-mpnet-base-v2'   # ĐÚNG tác giả (tiếng Anh)
RETR_OUT_DIR = '/content/drive/MyDrive/tkgqa_outputs_llama2_faithful'
os.makedirs(RETR_OUT_DIR, exist_ok=True)
RETRIEVER_FT_PATH = f'{RETR_OUT_DIR}/time_aware_retriever_fullft'

MAX_NEG_QUESTIONS = 8000   # số câu train dùng để đào negative (tăng nếu muốn)

if not os.path.exists(os.path.join(RETRIEVER_FT_PATH, 'modules.json')):
    _tr = list(train_questions); random.shuffle(_tr); _tr = _tr[:MAX_NEG_QUESTIONS]
    _queries = [f"{get_q(q)} {gold_answers(q)}" for q in _tr]
    _R = Retrieval(RETRIEVER_BASE, _queries, triple_list, None, None)
    _d, _c = _R.compute_similarity(n=15)
    _res = _R.get_result(_d, _c, _queries, re_rank=False)
    del _R; torch.cuda.empty_cache()
    neg_data = []
    for i, item in enumerate(tqdm(_tr, desc='Build negatives')):
        triples = _res[i].get('triple') or []
        if not triples: continue
        ql = get_q(item).lower()
        if 'first' in ql:  positive = sorted(triples, key=_sort_key)[0]
        elif 'last' in ql: positive = sorted(triples, key=_sort_key, reverse=True)[0]
        else:              positive = triples[0]
        negs = gen_negatives_author(positive, get_q(item), gold_answers(item))
        if not negs: continue
        neg_data.append({'question': get_q(item),
                         'positive': triple_to_text(positive),
                         'negative': [triple_to_text(n) for n in negs]})
    print(f'✅ {len(neg_data)} câu có negative')
    json.dump(neg_data, open(f'{RETR_OUT_DIR}/negatives.json', 'w', encoding='utf-8'), ensure_ascii=False)
else:
    neg_data = None
    print('✅ Đã có retriever fine-tune → bỏ qua đào negative')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — FULL FINE-TUNE TIME-AWARE RETRIEVER (bám fine_tuned_retriever.py)
#   all-mpnet-base-v2 + [B_SEP][A_SEP][F_SEP][L_SEP] + 4017 [Time_i]  (y hệt tác giả)
#   ContrastiveTensionLossInBatchNegatives(scale=1, dot) · 2 epoch · lr 5e-5 · warmup 1000
#   ★ FULL fine-tune, KHÔNG LoRA — đúng tác giả (mpnet 110M chạy tốt trên GPU phổ thông).
# ═══════════════════════════════════════════════════════════
from sentence_transformers import InputExample, losses, util, models
from torch.utils.data import DataLoader

if not os.path.exists(os.path.join(RETRIEVER_FT_PATH, 'modules.json')):
    _word = models.Transformer(RETRIEVER_BASE, max_seq_length=128)
    _tokens = ['[B_SEP]', '[A_SEP]', '[F_SEP]', '[L_SEP]'] + [f'[Time_{i}]' for i in range(4017)]
    _word.tokenizer.add_tokens(_tokens, special_tokens=True)   # 4017 time-token ICEWS (inert với VI nhưng giữ để trung thực)
    _word.auto_model.resize_token_embeddings(len(_word.tokenizer))
    _pool = models.Pooling(_word.get_word_embedding_dimension())
    retriever = SentenceTransformer(modules=[_word, _pool], device='cuda')

    train_data = []
    for ex in neg_data:
        train_data.append(InputExample(texts=[ex['question'], ex['positive']], label=1.0))
        for neg in ex['negative']:
            train_data.append(InputExample(texts=[ex['question'], neg], label=0.0))
    random.shuffle(train_data)

    BATCH = 64   # tác giả 256; giảm cho GPU phổ thông (KHÔNG đổi phương pháp)
    dl = DataLoader(train_data, shuffle=True, batch_size=BATCH)
    loss = losses.ContrastiveTensionLossInBatchNegatives(retriever, scale=1, similarity_fct=util.dot_score)
    print(f'Full fine-tune retriever: {len(train_data)} ví dụ · 2 epoch · lr 5e-5 · warmup 1000...')
    retriever.fit(train_objectives=[(dl, loss)], epochs=2, warmup_steps=1000,
                  optimizer_params={'lr': 5e-5}, show_progress_bar=True)
    retriever.save(RETRIEVER_FT_PATH)
    print(f'✅ Đã lưu retriever full-FT: {RETRIEVER_FT_PATH}')
    del retriever; import gc; gc.collect(); torch.cuda.empty_cache()
else:
    print(f'✅ Retriever full-FT sẵn có: {RETRIEVER_FT_PATH}')

RETRIEVER_PATH = RETRIEVER_FT_PATH
print('✅ Retriever dùng cho pipeline:', RETRIEVER_PATH)


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — PROMPT (đúng main.py) + Alpaca (đúng train.py) + REWRITE GPT-3.5
# ═══════════════════════════════════════════════════════════
def build_rewrite_prompt(fact, question):
    # ĐÚNG prompt main.py (gpt-3.5-turbo-0125); đã bỏ 1 ký tự thừa do lỗi gõ của tác giả
    return ('Replace facts in questions with explicit information from provided facts '
            'without any explanation.If you are not sure about the answer, output the '
            'original question. For instance, from the fact "Herman Van Rompuy replaced '
            'by Donald Tusk from 2009 to 2014." modify the question "what was donald '
            'tusk\'s position before he was replaced by herman van rompuy?" to "what was '
            'donald tusk\'s position before 2009?" Here is your turn: '
            f'Question: {question} Fact:{fact}.')

def build_reasoning_prompt(facts, question):
    # ĐÚNG prompt reasoning main.py (chú ý khoảng trắng sau \n và trước {question})
    return ('Based on the facts, please answer the given question. Keep the answer as '
            'simple as possible and return all the possible answers as a list.\n'
            f' Facts:{facts}\nQuestion:\n {question}?')

def _alpaca(instruction):
    # ĐÚNG PROMPT_DICT['prompt_no_input'] train.py
    return ('Below is an instruction that describes a task. Write a response that '
            'appropriately completes the request.\n\n### Instruction:\n' + instruction +
            '\n\n### Response:')

def _llm_gen(instruction, max_new_tokens=64, use_adapter=True, max_in=1500):
    text = _alpaca(instruction)   # raw Alpaca text, KHÔNG dùng chat template (đúng train.py)
    inp = tokenizer(text, return_tensors='pt', truncation=True, max_length=max_in,
                    add_special_tokens=True).to(DEVICE)
    ctx = (llm.disable_adapter() if (not use_adapter and hasattr(llm, 'disable_adapter')) else nullcontext())
    with torch.no_grad(), ctx:
        out = llm.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                           eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()

# ── Rewrite qua OpenAI GPT-3.5-turbo-0125 (đúng tác giả) ──
_OPENAI_MODEL = 'gpt-3.5-turbo-0125'
try:
    from openai import OpenAI as _OpenAI
    _oa = _OpenAI() if os.environ.get('OPENAI_API_KEY') else None
except Exception as _e:
    _oa = None; print('⚠️ openai chưa cài:', str(_e)[:60])

def _gpt_rewrite(prompt, max_retry=5):
    import time as _t; bo = 1.0
    for _ in range(max_retry):
        try:
            r = _oa.chat.completions.create(model=_OPENAI_MODEL, temperature=1.0,
                    messages=[{'role': 'user', 'content': prompt}])
            return r.choices[0].message.content.strip()
        except Exception as e:
            print('OpenAI lỗi, thử lại:', str(e)[:70]); _t.sleep(bo); bo *= 1.5
    return ''

def call_llm(prompt):
    if USE_OPENAI_REWRITE:
        assert _oa is not None, '❌ Cần OPENAI_API_KEY (hoặc USE_OPENAI_REWRITE=False).'
        return _gpt_rewrite(prompt).split(chr(10))[0].strip()
    return _llm_gen(prompt, use_adapter=False).split(chr(10))[0].strip()

USE_FT = True
def gen_answer(prompt, max_new_tokens=64):
    # trả TOÀN BỘ output để metric khớp predict_answer.py (không chỉ lấy dòng đầu)
    return _llm_gen(prompt, max_new_tokens=max_new_tokens, use_adapter=USE_FT)

print('✅ Prompt + gen (Alpaca) + rewrite (GPT-3.5) sẵn sàng')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — DỰNG DATA FINE-TUNE REASONING (bám main.py --type train)
#   retrieve(n=15, re_rank=False) → rewrite (GPT-3.5) → retrieve(n=15) → prompt + target
#   ⛔ KHÔNG rerank (đúng tác giả). target = str(list đáp án).
# ═══════════════════════════════════════════════════════════
N_TRAIN = 4000   # tác giả dùng toàn bộ train; đặt None để lấy hết (tốn API rewrite nhiều hơn)

def gold_list_str(item):
    g = gold_answers(item); return '[' + ', '.join(f"'{x}'" for x in g) + ']'

pool = [q for q in train_questions if get_gold(q) not in (None, '?', '', [])]
random.shuffle(pool)
if N_TRAIN: pool = pool[:N_TRAIN]
print(f'Dựng data fine-tune từ {len(pool)} câu train...')

# (1) retrieve lần 1 trên câu gốc → lấy fact hàng đầu cho bước rewrite
q1 = [get_q(q) for q in pool]
R1 = Retrieval(RETRIEVER_PATH, q1, triple_list, None, None)
d1, c1 = R1.compute_similarity(n=15)
bg = R1.get_result(d1, c1, q1, re_rank=False); del R1; torch.cuda.empty_cache()

# (2) rewrite train (đúng tác giả) — cache vào Drive để chỉ tốn API 1 lần
_RW_CACHE = f'{RETR_OUT_DIR}/train_rewritten_{len(pool)}.json'
if REWRITE_TRAIN:
    if os.path.exists(_RW_CACHE):
        rew = json.load(open(_RW_CACHE, encoding='utf-8')); print('✅ Nạp rewrite train từ cache')
    else:
        rew = []
        for i in tqdm(range(len(pool)), desc='Rewrite train (GPT-3.5)'):
            f0 = (bg[i].get('fact') or [''])[0]
            r = call_llm(build_rewrite_prompt(f0, q1[i]))
            rew.append(r if r else q1[i])
        json.dump(rew, open(_RW_CACHE, 'w', encoding='utf-8'), ensure_ascii=False)
        print('✅ Đã rewrite + cache train')
else:
    rew = q1[:]; print('⚠️ REWRITE_TRAIN=False → dùng câu gốc (kém trung thực)')

# (3) retrieve lần 2 trên câu đã rewrite → 15 fact đưa vào prompt
R2 = Retrieval(RETRIEVER_PATH, rew, triple_list, None, None)
d2, c2 = R2.compute_similarity(n=15)
fl = R2.get_result(d2, c2, rew, re_rank=False); del R2; torch.cuda.empty_cache()

train_examples = []
for i, item in enumerate(pool):
    facts = (fl[i].get('fact') or [])[:15]   # top-15, KHÔNG rerank
    if not facts: continue
    train_examples.append({'prompt': build_reasoning_prompt(facts, rew[i]),
                           'target': gold_list_str(item)})
print(f'✅ {len(train_examples)} cặp train')
print('PROMPT:', train_examples[0]['prompt'][:200], '...')
print('TARGET:', train_examples[0]['target'])


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — ★ QLoRA FINE-TUNE (điểm thay full-FT của tác giả)
#   Alpaca + mask phần prompt (-100) + target+eos · 3 epoch · MAX_LEN 1024 (đúng train.py)
#   lr 1e-4 cho QLoRA (tác giả full-FT lr 2e-5). 2e-4 dễ làm LLaMA2 sụp đổ → dùng 1e-4.
# ═══════════════════════════════════════════════════════════
MAX_LEN = 1024; EPOCHS = 3

class SFTDataset(torch.utils.data.Dataset):
    def __init__(self, ex):
        self.rows = []
        for e in ex:
            src = _alpaca(e['prompt'])
            pids = tokenizer(src, add_special_tokens=True)['input_ids']
            tids = tokenizer(e['target'] + tokenizer.eos_token, add_special_tokens=False)['input_ids']
            ids = (pids + tids)[:MAX_LEN]
            lab = ([-100] * len(pids) + tids)[:MAX_LEN]
            if len(ids) <= len(pids): continue
            self.rows.append({'input_ids': ids, 'labels': lab})
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def collate(b):
    ml = max(len(x['input_ids']) for x in b); pad = tokenizer.pad_token_id
    ids, lab, am = [], [], []
    for x in b:
        d = ml - len(x['input_ids'])
        ids.append(x['input_ids'] + [pad] * d); lab.append(x['labels'] + [-100] * d)
        am.append([1] * len(x['input_ids']) + [0] * d)
    return {'input_ids': torch.tensor(ids), 'labels': torch.tensor(lab), 'attention_mask': torch.tensor(am)}

OUT_DIR = RETR_OUT_DIR
LORA_DIR = f'{OUT_DIR}/llama2_qlora_reasoning'
FORCE_RETRAIN = True
import shutil as _sh
if FORCE_RETRAIN and os.path.exists(LORA_DIR): _sh.rmtree(LORA_DIR); print('🗑️ Xoá adapter cũ')

ds = SFTDataset(train_examples)
print('Dataset:', len(ds), 'mẫu')
args = TrainingArguments(output_dir='/content/ckpt',
    per_device_train_batch_size=2, gradient_accumulation_steps=8,   # batch hiệu dụng 16
    num_train_epochs=EPOCHS, learning_rate=1e-4, warmup_ratio=0.03,
    logging_steps=20, save_strategy='no', bf16=USE_BF16, fp16=(not USE_BF16),
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    optim='paged_adamw_8bit', report_to='none', lr_scheduler_type='cosine')
llm.config.use_cache = False
print(f'🚀 QLoRA fine-tune: {EPOCHS} epoch, {"bf16" if USE_BF16 else "fp16"}...')
Trainer(model=llm, args=args, train_dataset=ds, data_collator=collate).train()
llm.save_pretrained(LORA_DIR); print('✅ Lưu adapter:', LORA_DIR)

# ── Kiểm tra sụp đổ (5 câu khác nhau; nếu ra y hệt nhau → model hỏng) ──
try: llm.gradient_checkpointing_disable()
except Exception: pass
llm.config.use_cache = True; llm.eval()
_s = random.sample(train_examples, min(5, len(train_examples)))
_o = [_llm_gen(e['prompt'], use_adapter=True) for e in _s]
for x in _o: print('→', repr(x)[:80])
print('⚠️ SỤP ĐỔ — giảm lr về 5e-5 hoặc EPOCHS=1 rồi xoá adapter, chạy lại'
      if len(set(_o)) == 1 else f'✅ OK: {len(set(_o))}/5 đáp án khác nhau')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — PIPELINE TimeR4 đầy đủ: Retrieve → Rewrite → Retrieve → Reasoning
#   re_rank=False ở CẢ HAI lần retrieve (đúng main.py). Không rerank qtype.
# ═══════════════════════════════════════════════════════════
def run_pipeline(qs_raw, triples, retriever_path, label='', checkpoint_path=None):
    qs = copy.deepcopy(qs_raw)
    print(f'\n{"="*56}\n {label}\n{"="*56}')
    q1 = [get_q(q) for q in qs]

    print('[1/4] Retrieve (re_rank=False)...')
    R1 = Retrieval(retriever_path, q1, triples, None, None)
    d1, c1 = R1.compute_similarity(n=15)
    bg = R1.get_result(d1, c1, q1, re_rank=False); del R1; torch.cuda.empty_cache()

    print('[2/4] Rewrite (GPT-3.5)...')
    for i in tqdm(range(len(qs)), leave=False):
        f0 = (bg[i].get('fact') or [''])[0]
        r = call_llm(build_rewrite_prompt(f0, q1[i]))
        qs[i]['question'] = r if r else q1[i]

    print('[3/4] Retrieve lại trên câu đã rewrite (re_rank=False)...')
    q2 = [get_q(q) for q in qs]
    R2 = Retrieval(retriever_path, q2, triples, None, None)
    d2, c2 = R2.compute_similarity(n=15)
    fl = R2.get_result(d2, c2, q2, re_rank=False); del R2; torch.cuda.empty_cache()

    print('[4/4] Reasoning (LLaMA2 QLoRA)...')
    results, correct = [], 0
    for i, item in enumerate(tqdm(qs, leave=False)):
        facts = (fl[i].get('fact') or [])[:15]
        gold = get_gold(qs_raw[i])
        pred = gen_answer(build_reasoning_prompt(facts, get_q(item)))
        ok = evaluate_hit(pred, gold); correct += ok
        results.append({'question': get_q(qs_raw[i]), 'gold': str(gold), 'predicted': pred,
                        'parsed': parse_list_answer(pred), 'correct': ok, 'retrieved_facts': facts})
        if checkpoint_path and (i + 1) % 500 == 0:
            json.dump(results, open(checkpoint_path, 'w', encoding='utf-8'), ensure_ascii=False)
            print(f'  💾 {i+1}/{len(qs)} (đúng {correct})')
    em = correct / len(results) * 100
    print(f'✅ {label}: {correct}/{len(results)} = {em:.2f}%')
    return results, em

print('✅ Pipeline sẵn sàng')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — CHẠY TOÀN BỘ TEST + ablation (base vs QLoRA)
# ═══════════════════════════════════════════════════════════
try: llm.gradient_checkpointing_disable()
except Exception: pass
llm.config.use_cache = True; llm.eval()

TEST_N = len(test_questions); ABLATION_N = 200
_CKPT = f'{OUT_DIR}/results_faithful_ckpt.json'

USE_FT = True
res_ft, em_ft = run_pipeline(questions_1000[:TEST_N], triple_list, RETRIEVER_PATH,
                             label=f'LLaMA2 QLoRA (n={TEST_N})', checkpoint_path=_CKPT)
json.dump(res_ft, open(f'{OUT_DIR}/results_faithful.json', 'w', encoding='utf-8'),
          ensure_ascii=False, indent=2)

USE_FT = False
res_b, em_b = run_pipeline(questions_1000[:ABLATION_N], triple_list, RETRIEVER_PATH,
                           label=f'LLaMA2 BASE (n={ABLATION_N}, ablation)')
USE_FT = True
em_ft_sub = sum(x['correct'] for x in res_ft[:ABLATION_N]) / ABLATION_N * 100
print(f'\nBASE {em_b:.2f}% | QLoRA {em_ft_sub:.2f}% | Δ fine-tune {em_ft_sub-em_b:+.2f}')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 11 — BẢNG TỔNG KẾT
# ═══════════════════════════════════════════════════════════
print('═'*62)
print('  TimeR4 TRUNG THỰC (chỉ full-FT→QLoRA cho LLM) — Wikidata-VI')
print('═'*62)
print(f'  Retriever : all-mpnet-base-v2 FULL-FT (đúng tác giả)')
print(f'  LLM       : LLaMA2-7B QLoRA (thay full-FT)')
print(f'  Rewrite   : GPT-3.5-turbo-0125 (đúng tác giả)')
print(f'  {"LLaMA2 QLoRA (toàn test)":<32} {em_ft:>7.2f}%')
print(f'  {"vs v12 Qwen (45.39%)":<32} {em_ft-45.39:>+7.2f}')
print('═'*62)

summary = {
    'dataset': 'Wikidata-VI',
    'method': 'TimeR4 faithful (author released code) + QLoRA on LLM only',
    'retriever': 'all-mpnet-base-v2 full fine-tune (author)',
    'llm': 'LLaMA2-7B-chat QLoRA 4-bit',
    'rewrite': 'gpt-3.5-turbo-0125',
    'rerank': 're_rank=False (author)',
    'em_full_test': round(em_ft, 2),
    'em_base_subset': round(em_b, 2),
    'em_qlora_subset': round(em_ft_sub, 2),
    'test_n': TEST_N,
    'em_v12_qwen': 45.39,
    'delta_vs_v12': round(em_ft - 45.39, 2),
    'n_triples': len(triple_list),
}
json.dump(summary, open(f'{OUT_DIR}/summary_faithful.json', 'w', encoding='utf-8'),
          ensure_ascii=False, indent=2)
print('✅ Đã lưu summary_faithful.json')
